# 🌐 Notebook 1 — Découverte de l'API
**Masterclass MCP · AI4Africa Ignition · 07 mai 2026**

> **Objectif (10 min) :** comprendre ce qu'est une API et faire notre tout premier appel météo en Python.

À la fin de ce notebook, vous saurez :
- Ce qu'est une API et pourquoi on en a besoin
- Faire une requête HTTP avec `requests`
- Lire une réponse JSON
- Récupérer les coordonnées GPS d'une ville, puis sa météo actuelle


## 1. Qu'est-ce qu'une API ?

**API** = *Application Programming Interface*. C'est une porte d'entrée qu'un service met à disposition
pour qu'un programme puisse lui parler, sans passer par un site web ou une interface graphique.

Concrètement, pour la météo :
- On envoie une requête HTTP à une URL (ex : `https://api.open-meteo.com/...`)
- Le serveur répond avec des données, généralement au format **JSON**
- On utilise ces données dans notre propre code

Aujourd'hui on utilise **Open-Meteo** : gratuite, sans inscription, sans clé API.
Deux endpoints nous intéressent :
- `geocoding-api.open-meteo.com` → transforme un nom de ville en coordonnées GPS
- `api.open-meteo.com` → transforme des coordonnées GPS en météo actuelle


In [ ]:
# On importe la bibliothèque requests : elle sert à faire des appels HTTP
import requests
import urllib3

# Sur certains PC Windows, le certificat SSL du store système n'est pas
# reconnu par Python et les appels HTTPS échouent (SSLCertVerificationError).
# On désactive donc la vérification + son avertissement pour cet atelier.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("requests importé avec succès !")


## 2. Premier appel : trouver une ville

On commence par le **géocodage** : donner un nom de ville, récupérer sa latitude/longitude.


In [ ]:
URL_GEOCODAGE = "https://geocoding-api.open-meteo.com/v1/search"

# Les paramètres de la requête : ce qu'on envoie au serveur
params = {
    "name": "Yaounde",   # la ville qu'on cherche
    "count": 1,          # un seul résultat, le plus pertinent
    "language": "fr",
    "format": "json",
}

reponse = requests.get(URL_GEOCODAGE, params=params, timeout=10, verify=False)

# Le code 200 veut dire "tout s'est bien passé"
print("Code de statut HTTP :", reponse.status_code)


## 3. Lire la réponse JSON

`reponse.json()` transforme le texte reçu en dictionnaire Python qu'on peut explorer.


In [ ]:
donnees = reponse.json()

# On affiche tout, brut, pour voir à quoi ça ressemble
donnees


In [ ]:
# Les résultats sont dans la clé "results" (une liste)
resultat = donnees["results"][0]

print("Nom       :", resultat["name"])
print("Pays      :", resultat["country"])
print("Latitude  :", resultat["latitude"])
print("Longitude :", resultat["longitude"])


## 4. Deuxième appel : la météo actuelle

Maintenant qu'on a des coordonnées GPS, on peut appeler l'API météo.


In [ ]:
URL_METEO = "https://api.open-meteo.com/v1/forecast"

params_meteo = {
    "latitude": resultat["latitude"],
    "longitude": resultat["longitude"],
    "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m,weather_code",
    "timezone": "auto",
    "forecast_days": 1,
}

reponse_meteo = requests.get(URL_METEO, params=params_meteo, timeout=10, verify=False)
print("Code de statut HTTP :", reponse_meteo.status_code)

donnees_meteo = reponse_meteo.json()
donnees_meteo


## 5. Extraire les infos utiles

Les données actuelles se trouvent dans la clé `"current"`.


In [ ]:
actuel = donnees_meteo["current"]

print(f"Météo à {resultat['name']}, {resultat['country']}")
print(f"Température : {actuel['temperature_2m']} °C (ressentie {actuel['apparent_temperature']} °C)")
print(f"Humidité    : {actuel['relative_humidity_2m']} %")
print(f"Vent        : {actuel['wind_speed_10m']} km/h")
print(f"Code météo  : {actuel['weather_code']}  (à décoder — voir Notebook 2)")
print(f"Heure locale: {actuel['time']}")


## 🎯 À vous de jouer

Changez `"Yaounde"` par une autre ville (Dakar, Abidjan, Douala, Paris...) dans la cellule de la section 2,
puis relancez toutes les cellules dans l'ordre (`Cell → Run All` ou `Kernel → Restart & Run All`).

## ✅ Ce qu'on a appris

- Une API s'appelle avec une simple requête HTTP (`requests.get`)
- La réponse est du JSON, qu'on manipule comme un dictionnaire Python
- Il faut **deux appels** pour avoir la météo d'une ville : géocodage puis météo

## ➡️ Prochaine étape

Direction **`02_fonctions_python.ipynb`** pour transformer ce code brut en fonctions propres et réutilisables.
